# Pandas Series — Deep Dive Notes
This notebook covers Series basics, with detailed comments on every cell so you understand *why* something happens, not just the output.

In [1]:
# Importing Series and DataFrame classes directly from pandas
# so we can write 'Series(...)' instead of 'pd.Series(...)' every time
from pandas import Series, DataFrame
import pandas as pd   # 'pd' is the standard alias for the whole pandas library
import numpy as np    # needed because a Series stores its data internally as a numpy array

## What is a Series?
- A Series is a **1-D (one dimensional)** array-like object.
- It has two parts:
  1. **values** — the actual data (as a numpy array)
  2. **index** — a label for each value (like a dictionary 'key')
- Think of a Series as a single Excel column, where every row also has a name (the index).

In [2]:
# Passing only a list of numbers, no index specified
# When you don't give an index, pandas auto-creates 0,1,2,3... (a RangeIndex)
obj = Series([4,7,-5,3])
obj
# Reading the output:
# left side (0,1,2,3) = index (position/label)
# right side (4,7,-5,3) = actual values
# 'dtype: int64' tells us the numbers inside are 64-bit integers

0    4
1    7
2   -5
3    3
dtype: int64

### `.values` and `.index` attributes
Pandas gives you two attributes to pull out the two parts of a Series separately:
- `obj.values` → just the data (as a numpy array), no index
- `obj.index` → just the index object, no data
Internally, a Series is exactly these two things combined: values + index.

In [3]:
# .values attribute -> returns just the raw data as a numpy array
# no index shown here, since this is only the 'values' part
obj.values

array([ 4,  7, -5,  3])

In [4]:
# .index attribute -> returns the Series' index object
# we got RangeIndex(start=0, stop=4, step=1) because we didn't provide a custom index
# this means index starts at 0, goes up to (but not including) 4, in steps of 1
obj.index

RangeIndex(start=0, stop=4, step=1)

## Giving a custom index
Instead of the default 0,1,2,3 index, we can give our own labels (like 'd','e','f','g').
This is useful when you want every value to have a meaningful name (similar to dictionary keys).

In [5]:
# passing our own list of labels via the 'index' parameter
# now instead of position (0,1,2,3), the index becomes 'd','e','f','g'
# NOTE: order must match -> d->4, e->7, f->-5, g->3
obj2 = Series([4,7,-5,3], index=['d','e','f','g'])
obj2

d    4
e    7
f   -5
g    3
dtype: int64

In [6]:
# dtype='object' now because the index is custom (labels are strings)
# 'Index' is a generic index object (as opposed to RangeIndex when labels are custom)
obj2.index

Index(['d', 'e', 'f', 'g'], dtype='str')

In [7]:
# values stay the same whether the index is default or custom -> just the array of numbers
obj2.values

array([ 4,  7, -5,  3])

## Accessing values by index (like a dictionary lookup)
You can access a Series like a dictionary: `obj2['label']`
Internally, it looks up the label and returns its corresponding value — feels like a fast O(1) lookup.

In [8]:
# getting the value at label 'd' -> 4
# note: result shows as 'np.int64(4)' since numpy's int64 dtype is used internally
obj2['d']

np.int64(4)

In [9]:
# similarly, the value at label 'e' -> 7
obj2['e']

np.int64(7)

In [10]:
# value at label 'f' -> -5
obj2['f']

np.int64(-5)

In [11]:
# value at label 'g' -> 3
obj2['g']

np.int64(3)

## Updating a value
A Series is mutable — meaning even after creation you can change its values, just like a dictionary.

In [12]:
# changing the value at index 'd' from 4 to 9 (in-place update)
# this works exactly like a dict: obj2['d'] = 9 -> old value 4 gets overwritten
obj2['d']=9
obj2

d    9
e    7
f   -5
g    3
dtype: int64

## Boolean filtering (applying a condition on a Series)
`obj2 > 0` creates a Boolean Series (True/False for every index), and `obj2[boolean_series]`
keeps only the values where the condition is True. This is called 'masking'.

In [13]:
# obj2>0 -> internally first creates a Boolean Series: d(9)->True, e(7)->True, f(-5)->False, g(3)->True
# then obj2[boolean_series] keeps only the True-indexed values, the rest get dropped
obj2[obj2>0]
# -5 value will remove   <- the 'f' row is gone because -5 > 0 was False

d    9
e    7
g    3
dtype: int64

## Vectorized operations (NumPy-style broadcasting)
You can apply math operations directly on a Series without writing a loop —
the operation gets applied element-wise (this is inherited from NumPy).

In [14]:
# multiplied every value by 2 - a single line operates on the whole Series
# no for-loop needed, this is a vectorized operation (fast + clean)
obj2*2

d    18
e    14
f   -10
g     6
dtype: int64

In [15]:
# np.exp() -> computes the exponential (e^x) of every value, element-wise
# this shows numpy's mathematical functions also work directly on a Series
np.exp(obj2)

d    8103.083928
e    1096.633158
f       0.006738
g      20.085537
dtype: float64

## Membership check (the `in` keyword)
The `in` keyword on a Series checks only the **index labels**, not the values!
This is exactly like a dictionary, where `'key' in dict` checks the keys.

In [16]:
# is 'd' present in the index? -> True (since 'd' is one of obj2's labels)
'd' in obj2

True

In [17]:
# is 'a' present in the index? -> False (there's no label called 'a' in obj2)
'a' in obj2

False

## Creating a Series from a Python dictionary
You can build a Series directly from a Python dict.
- dict **keys** → become the Series index
- dict **values** → become the Series data

In [18]:
# a dictionary mapping state -> population
sdata = {'Ohio': 35000, 'Texas': 71000, 'Oregon': 16000, 'Utah': 5000}
# Series(sdata) -> pandas automatically turns dict keys into the index and values into the data
obj3=Series(sdata)
obj3

Ohio      35000
Texas     71000
Oregon    16000
Utah       5000
dtype: int64

**Note:** When you only pass a dict (without an explicit index), the resulting Series' index
comes from the dict's keys (modern pandas preserves insertion order; older pandas used sorted order).

## Dict + custom index list (handling mismatches)
If you also provide your own index list (one that doesn't match the dict's keys exactly),
something interesting happens:
- labels that ARE found in the dict get their value
- labels that are NOT found in the dict get **NaN** (missing value)
- and any dict keys NOT present in your new index list get **dropped**

In [19]:
# 'California' is new (wasn't in sdata) -> will get NaN
# 'Utah' was in sdata but is not in the new states list -> so Utah disappears
states= ['California', 'Ohio', 'Oregon', 'Texas']
obj4=Series(sdata,index=states)
obj4
# dtype is now float64 because NaN is a float value,
# and all values in one Series must share the same dtype, so the ints got upcast to float too

California        NaN
Ohio          35000.0
Oregon        16000.0
Texas         71000.0
dtype: float64

## Detecting missing data: `isnull()` / `notnull()`
Missing values (NaN) are common in real-world data. Pandas gives two handy functions:
- `pd.isnull(series)` → True wherever the value is missing (NaN)
- `pd.notnull(series)` → True wherever a value is present (opposite of isnull)
You can also do this with the Series method version: `series.isnull()`.

In [20]:
# The isnull and notnull functions in pandas should be used to detect missing data:
# pd.isnull(obj4) -> checks each index for whether its value is NaN or not
# California -> True (it's NaN), everything else -> False (they have real values)
pd.isnull(obj4) # we can write in this way too :-obj4.isnull()

California     True
Ohio          False
Oregon        False
Texas         False
dtype: bool

In [21]:
# notnull() is the exact opposite of isnull()
# True where a value is present, False where it's NaN
pd.notnull(obj4)

California    False
Ohio           True
Oregon         True
Texas          True
dtype: bool

In [22]:
# printing obj3 again for reference (no missing values in this one)
obj3

Ohio      35000
Texas     71000
Oregon    16000
Utah       5000
dtype: int64

In [23]:
# looking at obj4 again - California is NaN, the other 3 states have values
obj4

California        NaN
Ohio          35000.0
Oregon        16000.0
Texas         71000.0
dtype: float64

## Index alignment: adding two Series (`obj3 + obj4`)
This is one of pandas' most powerful features — when you add two Series, pandas **automatically
aligns matching indexes**, even if their original order is different.
- Any index common to both Series → values get added
- Any index present in only one Series → result is NaN (since the other Series has no value for it)
This is similar to an SQL 'outer join'.

In [24]:
# obj3's index: Ohio, Texas, Oregon, Utah
# obj4's index: California, Ohio, Oregon, Texas
# common index (Ohio, Texas, Oregon) -> values get added
# Utah is only in obj3, California is only in obj4 -> both result in NaN
# (because NaN + any number = NaN, and anything missing from the other Series is treated as NaN too)
obj5=obj3+obj4
obj5

California         NaN
Ohio           70000.0
Oregon         32000.0
Texas         142000.0
Utah               NaN
dtype: float64

In [25]:
# California and Utah are NaN in obj5 (verifying with isnull)
pd.isnull(obj5)

California     True
Ohio          False
Oregon        False
Texas         False
Utah           True
dtype: bool

In [26]:
# adding directly and checking notnull in one line - same result as with obj5
pd.notnull(obj3+obj4)

California    False
Ohio           True
Oregon         True
Texas          True
Utah          False
dtype: bool

## Naming a Series and its Index (`.name`)
Both the Series itself and its index have their own separate `.name` attribute. This is mainly
for labeling/readability, especially useful when this Series later becomes a column in a
DataFrame — then this name acts like the column header.

In [27]:
# Both the Series object itself and its index have a name attribute, which integrates with other key areas of pandas functionality:
# obj4.name -> names the whole Series (like naming a 'population' column)
# obj4.index.name -> names just the index (like labeling it 'state')
obj4.name='poplutaion'
obj4.index.name='state'

In [28]:
# let's see how the Series looks after applying the names
# 'state' will appear above the index and 'poplutaion' will look like the column name for values
obj4

state
California        NaN
Ohio          35000.0
Oregon        16000.0
Texas         71000.0
Name: poplutaion, dtype: float64

---
### Quick Recap
- Series = values + index (a labeled 1-D array)
- `.values`, `.index` pull out the two parts separately
- Dictionary-like access: `obj['label']`, `'label' in obj`
- Boolean filtering: `obj[obj>0]`
- Vectorized math operations work directly on a Series
- Creating a Series from a dict: keys->index, values->data
- Adding two Series triggers **index alignment**, mismatches produce NaN
- Use `isnull()`/`notnull()` to detect missing data
- Use `.name` and `.index.name` for labeling